# Lab 10.4: Customers Like You (data case)

Chapter 10's case study. A recommender for a UK online gift retailer, built on the public **Online Retail II** dataset (Chen, 2012; UCI Machine Learning Repository, https://doi.org/10.24432/C5CG6D, CC BY 4.0). We build two recommenders, write the explanations each would show, and check them for faithfulness, privacy and influence. The first run downloads about 45 MB and converts the Excel file once (one or two minutes); later runs read the cached copy.

In [1]:
import os, pathlib, tempfile, urllib.request, zipfile, time
import numpy as np, pandas as pd
URL = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
cache = pathlib.Path(os.environ.get("XAI_DATA", pathlib.Path(tempfile.gettempdir()) / "xai-book-data"))
cache.mkdir(parents=True, exist_ok=True)
csv = cache / "online_retail_ii.csv.gz"
if not csv.exists():
    z = cache / "online_retail_ii.zip"
    if not z.exists():
        urllib.request.urlretrieve(URL, z)
    with zipfile.ZipFile(z) as f:
        f.extractall(cache)
    sheets = pd.read_excel(cache / "online_retail_II.xlsx", sheet_name=None, dtype={"Invoice": str, "StockCode": str})
    pd.concat(sheets.values(), ignore_index=True).to_csv(csv, index=False)
df = pd.read_csv(csv, dtype={"Invoice": str, "StockCode": str}, parse_dates=["InvoiceDate"])
print("rows:", len(df), " from", df.InvoiceDate.min().date(), "to", df.InvoiceDate.max().date())

rows: 1067371  from 2009-12-01 to 2011-12-09


**Checkpoint 1:** 1,067,371 rows from 2009-12-01 to 2011-12-09.

In [2]:
d = df.dropna(subset=["Customer ID"])
d = d[~d.Invoice.str.startswith("C")]                                   # drop cancellations
d = d[(d.Quantity > 0) & (d.Price > 0) & d.StockCode.str.match(r"^\d{5}")].drop_duplicates()   # products only
cut = pd.Timestamp("2011-09-01")
tr, te = d[d.InvoiceDate < cut], d[d.InvoiceDate >= cut]
ic = tr.groupby("StockCode")["Customer ID"].nunique()
items = ic[ic >= 2].index                                               # products bought by 2+ customers
tr = tr[tr.StockCode.isin(items)]
cust = tr["Customer ID"].unique()
ci = {c: k for k, c in enumerate(cust)}; ii = {s: k for k, s in enumerate(items)}
B = np.zeros((len(cust), len(items)), dtype=np.float32)                 # customer x product, 1 = bought
B[tr["Customer ID"].map(ci).values, tr.StockCode.map(ii).values] = 1
n, bs = B.sum(0), B.sum(1)
el = np.where(bs >= 5)[0]                                               # customers we recommend to
names = tr.drop_duplicates("StockCode").set_index("StockCode").Description
print("customers:", len(cust), " products:", len(items), " customers with 5+ products:", len(el))

customers: 5223  products: 4255  customers with 5+ products: 4931


**Checkpoint 2:** 5,223 customers, 4,255 products and 4,931 customers with at least five distinct products in the training period (before 1 September 2011).

## Part A. Two recommenders, two explanations

*Item-based* (in the spirit of item-to-item collaborative filtering, Linden et al., 2003): a product scores high if it is similar to products you bought; the explanation is "because you bought X". *User-based*: find your 10 most similar customers; recommend what most of them bought; the explanation is "m of the 10 customers most like you bought it".

In [3]:
C = B.T @ B; np.fill_diagonal(C, 0)                                     # co-purchase counts
S = C / np.sqrt(np.outer(n, n))                                         # cosine similarity of products
score = B @ S; score[B > 0] = -1
rec_item = score.argmax(1)
Un = B / np.sqrt(bs)[:, None]; U = Un @ Un.T; np.fill_diagonal(U, -1)   # cosine similarity of customers
nb = np.argsort(-U, 1)[:, :10]
rec_user, m_user, m_item = np.zeros(len(cust), int), [], []
for u in el:
    cnt = B[nb[u]].sum(0); cnt[B[u] > 0] = -1
    rec_user[u] = cnt.argmax(); m_user.append(cnt.max()); m_item.append(cnt[rec_item[u]])
m_user, m_item = np.array(m_user).astype(int), np.array(m_item).astype(int)

def explain_item(u):
    j = rec_item[u]; basket = np.where(B[u])[0]; i = basket[np.argmax(S[basket, j])]
    return f"Recommended: {names[items[j]]}. Because you bought {names[items[i]]} ({int(C[i, j])} other customers bought both)."
for u in el[:3]:
    print(explain_item(u))
print("user-based: neighbors behind the explanation (0..10):", np.bincount(m_user, minlength=11))

Recommended: PACK OF 72 RETRO SPOT CAKE CASES. Because you bought 72 SWEETHEART FAIRY CAKE CASES (374 other customers bought both).
Recommended: WHITE HANGING HEART T-LIGHT HOLDER. Because you bought LOVE BUILDING BLOCK WORD (365 other customers bought both).
Recommended: FELTCRAFT DOLL MOLLY. Because you bought FELTCRAFT DOLL ROSIE (227 other customers bought both).
user-based: neighbors behind the explanation (0..10): [  0  69 420 641 730 677 603 526 475 384 406]


**Checkpoint 3:** three readable item-based explanations, and the distribution 0, 69, 420, 641, 730, 677, 603, 526, 475, 384, 406 for how many of the 10 nearest customers back the user-based recommendation.

## Part B. Faithfulness: does the explanation come from the model that decided?

Suppose the shop runs the item-based recommender but, because it sounds friendlier, shows the text "customers like you bought this". How often is that sentence false?

In [4]:
print("item-based recommendations that none of the 10 most similar customers bought:",
      int((m_item == 0).sum()), f"({(m_item == 0).mean():.1%})")
print("... that at most 2 of them bought:", int((m_item <= 2).sum()), f"({(m_item <= 2).mean():.1%})")

item-based recommendations that none of the 10 most similar customers bought: 485 (9.8%)
... that at most 2 of them bought: 1718 (34.8%)


**Checkpoint 4:** 485 recommendations (9.8 percent) would carry a "customers like you" sentence that none of the customer's ten nearest neighbors supports.

## Part C. Privacy: how many people stand behind an explanation?

In [5]:
few = m_user < 3
print("user-based explanations backed by fewer than 3 neighbors:", int(few.sum()), f"({few.mean():.1%})")
print("... backed by exactly 1 neighbor:", int((m_user == 1).sum()))
country = tr.groupby("Customer ID").Country.first()
cc = country.value_counts()
print("countries with 5 or fewer customers:", int((cc <= 5).sum()), " customers in them:", int(cc[cc <= 5].sum()))

user-based explanations backed by fewer than 3 neighbors: 489 (9.9%)
... backed by exactly 1 neighbor: 69
countries with 5 or fewer customers: 21  customers in them: 43


**Checkpoint 5:** 489 explanations (9.9 percent) rest on one or two neighbors, 69 of them on exactly one; 21 countries have five or fewer customers (43 customers). An explanation such as "the one customer most like you in your country bought this" would disclose a real person's purchase.

## Part D. Influence: which customers shaped everyone else's recommendations?

For this model, removing one customer and retraining is exact and cheap: subtract that customer's co-purchases and recompute.

In [6]:
def loo_share(u):
    b = B[u]; C2 = C - np.outer(b, b); np.fill_diagonal(C2, 0); n2 = n - b
    den = np.sqrt(np.outer(n2, n2)); S2 = np.divide(C2, den, out=np.zeros_like(C2), where=den > 0)
    s2 = B @ S2; s2[B > 0] = -1; r2 = s2.argmax(1); others = el[el != u]
    return (r2[others] != rec_item[others]).mean()
rng = np.random.default_rng(0)
sample = np.r_[np.argsort(-bs)[:10], rng.choice(el, 30, replace=False)]
t0 = time.time()
share = np.array([loo_share(u) for u in sample])
print(f"40 retrainings in {time.time() - t0:.0f} s")
print("10 largest buyers, percent of other customers' top pick changed:", np.round(share[:10] * 100, 2))
print("30 random customers, median:", np.round(np.median(share[10:]) * 100, 2), " max:", np.round(share[10:].max() * 100, 2))

40 retrainings in 9 s
10 largest buyers, percent of other customers' top pick changed: [2.37 2.07 2.33 1.52 2.58 1.95 1.16 1.08 1.2  1.52]
30 random customers, median: 0.08  max: 0.79


**Checkpoint 6:** removing one of the ten largest buyers changes between about 1.1 and 2.6 percent of other customers' top recommendation; for a randomly chosen customer the median is about 0.08 percent, and only customers with hundreds of distinct products come near 1 percent.

## Part E. Does a privacy rule cost accuracy?

In [7]:
buys_later = te.groupby("Customer ID").StockCode.apply(set)
def hit_rate(rec):
    h = [items[rec[u]] in buys_later[cust[u]] for u in el if cust[u] in buys_later.index]
    return np.mean(h), len(h)
print("item-based, no rule:", hit_rate(rec_item))
S5 = S * (C >= 5); s5 = B @ S5; s5[B > 0] = -1
print("item-based, pairs bought by 5+ customers only:", hit_rate(s5.argmax(1)))
print("user-based:", hit_rate(rec_user))

item-based, no rule: (np.float64(0.08844133099824869), 2284)


item-based, pairs bought by 5+ customers only: (np.float64(0.08756567425569177), 2284)
user-based: (np.float64(0.0796847635726795), 2284)


**Checkpoint 7:** about 9 percent of the 2,284 eligible customers who bought again after 1 September 2011 bought the item recommended to them (item-based: 0.088). Restricting explanations to product pairs bought by at least five customers moves the hit rate from 0.088 to 0.088 (0.0884 to 0.0876); the user-based recommender scores about 0.080.

## Questions

1. Write the explanation you would show customer `el[0]`, and the one you would refuse to show. Why?
2. Part B: is the "customers like you" sentence a lie, a simplification or acceptable marketing? Argue one position.
3. Part D: should the largest wholesale buyers be told that their purchases shape other customers' recommendations? What would a data protection officer say?
4. Propose a suppression rule (threshold and fallback text) and state its cost using Parts C and E.